# 02 · EDA del abandono: la foto del asociado

**Diseño:** se observa a cada asociado en una fecha fija **t0 = corte − H** y se pregunta si se retira en los
H meses siguientes.

- **Población:** activos en t0 (los retirados antes de t0 ya no eran asociados en la foto).
- **Objetivo:** `#RETIROS` = 1 si se retira entre t0 y el corte.
- **Variables:** EDAD y ANTIGUEDAD calculadas a t0 (`abandono.foto` + `abandono.preparar`).

Con los 50 de `data/ejemplo` la foto a 3 meses deja 1 solo retiro: por eso este notebook usa 2.000 asociados
generados (semilla 7) si `DATOS_ASOCIADOS` tiene pocas filas.

In [ ]:
import plotly.express as px

from proyecto_ml import abandono as a
from proyecto_ml.config import get_settings
from proyecto_ml.generador import GeneradorAsociados

HORIZONTE = 3  # meses
COLORES = {"se queda": "#2a78d6", "se retira": "#eb6834"}  # paleta de evaluate.py, en orden fijo

df = a.cargar(get_settings().datos_asociados)
if len(df) < 500:
    print(f"{len(df)} filas en DATOS_ASOCIADOS: muy pocas para la foto. Se usan 2.000 generados (semilla 7).")
    df = GeneradorAsociados(2000, seed=7).generar()
print(f"{len(df)} asociados, corte {a.FECHA_CORTE.date()}")

## 1. ¿Cuántos quedan en la foto según el horizonte?

Con un horizonte corto entran casi todos, pero pocos se retiran. Con 12 meses la tasa se parece a la del
extracto completo. Es la decisión de negocio: ¿con cuánta anticipación queremos avisar?

In [ ]:
resumen = a.resumen_foto(df).reset_index()
display(resumen.round(3))
fig = px.bar(
    resumen,
    x="horizonte_meses",
    y="tasa_retiro",
    text=resumen["tasa_retiro"].map("{:.1%}".format),
    hover_data=["fecha_observacion", "asociados", "se_retiran"],
    labels={"horizonte_meses": "Horizonte (meses)", "tasa_retiro": "Tasa de retiro en la foto"},
    title="Tasa de retiro según el horizonte de la foto",
    color_discrete_sequence=[COLORES["se retira"]],
)
fig.update_xaxes(type="category")
fig.update_yaxes(tickformat=".0%")
fig.update_layout(template="plotly_white")
fig

## 2. La foto elegida: población y objetivo

In [ ]:
vista = a.foto(df, HORIZONTE)
X = a.preparar(vista)  # EDAD y ANTIGUEDAD a t0
clase = vista[a.TARGET].map({0: "se queda", 1: "se retira"}).rename("clase")
print(
    f"t0 = {a.fecha_observacion(HORIZONTE).date()} · {len(vista)} asociados activos · "
    f"{vista[a.TARGET].sum()} se retiran ({vista[a.TARGET].mean():.1%})"
)
clase.value_counts().to_frame("asociados")

## 3. Comportamiento con el fondo: promedio por clase

Los conteos de solicitudes son la señal principal. **Pregunta para Edwin:** ¿se miden a la fecha de extracción o
tienen fecha? Si son a la extracción, para los retirados incluyen lo que pasó después de t0 (fuga de información).

In [ ]:
promedios = X[a.COMPORTAMIENTO].groupby(clase).mean().T.rename_axis("variable").reset_index()
largo = promedios.melt(id_vars="variable", var_name="clase", value_name="promedio")
fig = px.bar(
    largo,
    y="variable",
    x="promedio",
    color="clase",
    barmode="group",
    orientation="h",
    color_discrete_map=COLORES,
    labels={"promedio": "Promedio por asociado", "variable": ""},
    title=f"Comportamiento promedio por clase (foto a {HORIZONTE} meses)",
    height=520,
)
fig.update_layout(template="plotly_white", legend_title_text="")
fig

## 4. Perfil: edad, antigüedad y salario por clase

In [ ]:
for columna in ["EDAD", "ANTIGUEDAD", "SALARIO"]:
    fig = px.box(
        X.assign(clase=clase),
        x="clase",
        y=columna,
        color="clase",
        color_discrete_map=COLORES,
        points=False,
        title=f"{columna} por clase (a t0)",
    )
    fig.update_layout(template="plotly_white", showlegend=False)
    fig.show()

## 5. Categóricas: tasa de retiro por categoría

In [ ]:
for columna in ["TIPOCONTRATO", "SEXO", "CARGO", "AGENCIA"]:
    tasa = vista.assign(**{columna: X[columna]}).groupby(columna)[a.TARGET].agg(["mean", "size"]).reset_index()
    fig = px.bar(
        tasa,
        x=columna,
        y="mean",
        hover_data={"size": True},
        labels={"mean": "Tasa de retiro", "size": "asociados"},
        title=f"Tasa de retiro por {columna}",
        color_discrete_sequence=[COLORES["se retira"]],
    )
    fig.update_yaxes(tickformat=".0%")
    fig.update_layout(template="plotly_white")
    fig.show()

## 6. Hallazgos y preguntas (para la presentación del 11-nov)

- **Diseño:** foto a t0 = corte − 3 meses. Con datos generados la tasa de retiro baja de 30 % a ~9 %, más cerca de
  un fondo real.
- **Con 50 filas no se puede:** queda 1 retiro. Se entrena con 2.000 generados hasta tener el extracto.
- **Preguntas para Edwin:**
  1. ¿Los conteos de solicitudes (ULT3MESES, cancelaciones...) tienen fecha, o son a la fecha de extracción?
  2. ¿Cuál es la tasa de retiro anual real del fondo?
  3. ¿Qué % de asociados tiene al menos una solicitud de cancelación o de disminución de cuotas?
  4. Mediana de antigüedad y de salario.